<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Algopro/AlgoPro_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [119]:
!gdown 168r-t-UMg4wyradNm5Sn3eg6tCyYZ-Bf

Downloading...
From: https://drive.google.com/uc?id=168r-t-UMg4wyradNm5Sn3eg6tCyYZ-Bf
To: /content/dates.jsonl
100% 7.14M/7.14M [00:00<00:00, 89.7MB/s]


In [120]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import random
from datetime import datetime, timedelta

In [121]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [122]:
import json
from collections import Counter

DATA_PATH = "dates.jsonl"

data = []
num_dates_dist = Counter()
index_dist = Counter()

with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        text = obj["text"]
        iso_date = obj["date"]
        idx = obj["index"]
        n_dates = obj["num_dates"]

        # prefix the text with the index
        prefixed = f"{idx}: {text}"
        data.append((prefixed, iso_date))

        num_dates_dist[n_dates] += 1
        index_dist[idx] += 1

random.shuffle(data)

print(f"Loaded {len(data)} examples")
print(f"\nDates per paragraph: {dict(sorted(num_dates_dist.items()))}")
print(f"Index distribution:  {dict(sorted(index_dist.items()))}")

print(f"\nSample (input -> target) pairs:")
for inp, target in data[:5]:
    preview = inp if len(inp) <= 150 else inp[:147] + "..."
    print(f"  {preview}\n    -> {target}\n")

lengths = [len(inp) for inp, _ in data]
print(f"Input length: min: {min(lengths)}, avg: {sum(lengths)/len(lengths):.1f}, max: {max(lengths)}")

Loaded 15000 examples

Dates per paragraph: {1: 4758, 2: 4834, 3: 3351, 4: 2057}
Index distribution:  {0: 8867, 1: 3956, 2: 1620, 3: 557}

Sample (input -> target) pairs:
  0: I discovered a well-preserved Roman mosaic beneath the ruins of an old villa on 25/07/1954. This find included 17 intact tiles depicting scenes ...
    -> 1954-07-25

  0: 12 AUG 1928 marked the day the Thompson family gathered at the old barn for the first time since the Great Depression, a moment preserved in a f...
    -> 1928-08-12

  1: I remember the sharp tang of copper from the old filing cabinet as I stared at the blueprints on 02-03-1959, the ink smudged where my thumb had ...
    -> 1953-11-01

  2: On Fri 28 Dec 2012, a formal business transaction was initiated between the two parties, coinciding with inclement winter weather that hindered ...
    -> 1908-10-28

  1: During a field study in the Andes in 2019/12/20, a pre-identified Inca stone tablet was examined for inscriptions that may reflect early

In [123]:
from collections import Counter
input_char_counts = Counter()
output_char_counts = Counter()
for inp, target in data:
    input_char_counts.update(inp)
    output_char_counts.update(target)

# keep all chars that appear at least 2 times
input_chars = sorted(c for c, n in input_char_counts.items() if n >= 2)
output_chars = sorted(output_char_counts.keys())

PAD, SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"

input_vocab = [PAD, UNK] + input_chars
output_vocab = [PAD, SOS, EOS] + output_chars

input_stoi = {c: i for i, c in enumerate(input_vocab)}
input_itos = {i: c for i, c in enumerate(input_vocab)}
output_stoi = {c: i for i, c in enumerate(output_vocab)}
output_itos = {i: c for i, c in enumerate(output_vocab)}

print(f"Input vocab size:  {len(input_vocab)}")
print(f"Output vocab size: {len(output_vocab)}")

# cap input length at 99th percentile to avoid outliers bloating padding
sorted_lens = sorted(len(inp) for inp, _ in data)
MAX_INPUT_LEN = sorted_lens[int(0.99 * len(sorted_lens))]
MAX_OUTPUT_LEN = len("YYYY-MM-DD") + 2  # +SOS +EOS
print(f"Max input length (99th pct): {MAX_INPUT_LEN}")
print(f"Max output length: {MAX_OUTPUT_LEN}")

# drop examples above the cap so we don't truncate actual dates
kept = [(inp, tgt) for inp, tgt in data if len(inp) <= MAX_INPUT_LEN]
print(f"Keeping {len(kept)}/{len(data)} examples after length cap")
data = kept

def encode_input(s):
    ids = [input_stoi.get(c, input_stoi[UNK]) for c in s]
    ids += [input_stoi[PAD]] * (MAX_INPUT_LEN - len(ids))
    return ids

def encode_output(s):
    ids = [output_stoi[SOS]] + [output_stoi[c] for c in s] + [output_stoi[EOS]]
    ids += [output_stoi[PAD]] * (MAX_OUTPUT_LEN - len(ids))
    return ids

class DateDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        inp, tgt = self.pairs[idx]
        return (torch.tensor(encode_input(inp), dtype=torch.long),
                torch.tensor(encode_output(tgt), dtype=torch.long),
                inp, tgt)

split = int(0.9 * len(data))
train_data = data[:split]
test_data = data[split:]

train_loader = DataLoader(DateDataset(train_data), batch_size=64, shuffle=True,
                          collate_fn=lambda b: (torch.stack([x[0] for x in b]),
                                                torch.stack([x[1] for x in b]),
                                                [x[2] for x in b],
                                                [x[3] for x in b]))
test_loader = DataLoader(DateDataset(test_data), batch_size=64,
                         collate_fn=lambda b: (torch.stack([x[0] for x in b]),
                                               torch.stack([x[1] for x in b]),
                                               [x[2] for x in b],
                                               [x[3] for x in b]))

print(f"\nTrain size: {len(train_data)}, Test size: {len(test_data)}")

Input vocab size:  125
Output vocab size: 14
Max input length (99th pct): 763
Max output length: 12
Keeping 14853/15000 examples after length cap

Train size: 13367, Test size: 1486


In [124]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (batch, n_queries, d_k)       - queries
    K: (batch, n_keys, d_k)          - keys
    V: (batch, n_keys, d_v)          - values
    mask: (batch, n_queries, n_keys) - 1 where valid, 0 where padded

    Returns:
      output:     (batch, n_queries, d_v)
      attn_weights: (batch, n_queries, n_keys)
    """
    d_k = Q.size(-1)
    scale = d_k ** 0.5

    # step 1: compute raw scores Q @ K^T
    # shape: (batch, n_queries, n_keys)
    scores = Q @ K.transpose(-1, -2)

    # step 2: scale by sqrt(d_k) for softmax
    scores /= scale

    # step 3: apply mask (set padded positions to -inf so softmax ignores them)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    # step 4: softmax over the keys dimension -> attention weights
    attn_weights = torch.softmax(scores, dim=-1)

    # step 5: weighted sum of values
    output = attn_weights @ V

    return output, attn_weights


# check on tiny inputs
Q_test = torch.tensor([[[1.0, 0.0], [0.0, 1.0]]]) # 2 queries
K_test = torch.tensor([[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]]) # 3 keys
V_test = torch.tensor([[[10.0, 0.0], [0.0, 20.0], [5.0, 5.0]]]) # 3 values
mask = torch.zeros((1, 2, 3))
mask[:, 0, :2] = 1

out, w = scaled_dot_product_attention(Q_test, K_test, V_test, mask)
print(f"Attention weights shape: {w.shape}")
print(f"Attention weights:\n{w}")
print(f"Output:\n{out}")

Attention weights shape: torch.Size([1, 2, 3])
Attention weights:
tensor([[[0.6698, 0.3302, 0.0000],
         [   nan,    nan,    nan]]])
Output:
tensor([[[6.6976, 6.6048],
         [   nan,    nan]]])


In [130]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, 1, batch_first=True, bidirectional=True)
        self.proj = nn.Linear(2*hidden_dim, hidden_dim) # merge bidirectional

    def forward(self, x):
        x = self.embedding(x)
        z, _ = self.lstm(x)
        return self.proj(z)

class AttentionDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm_cell = nn.LSTMCell(embed_dim + hidden_dim, hidden_dim)

        # linear projections to produce Q, K, V for attention
        self.W_q = nn.Linear(hidden_dim, hidden_dim)
        self.W_k = nn.Linear(hidden_dim, hidden_dim)
        self.W_v = nn.Linear(hidden_dim, hidden_dim)

        self.out_proj = nn.Linear(hidden_dim * 2, vocab_size)

    def forward(self, encoder_outputs, input_mask, target=None, max_len=MAX_OUTPUT_LEN):
        """
        encoder_outputs: (B, T_in, hidden_dim)
        input_mask: (B, T_in) - 1 for real tokens, 0 for padding
        target: (B, T_out) - ground-truth output for teacher forcing (training)
        """
        B, T_in, H = encoder_outputs.shape

        # pre-compute K and V from encoder outputs (they don't change per decoder step)
        k, v = self.W_k(encoder_outputs), self.W_v(encoder_outputs)

        # initialize decoder hidden state
        h = torch.zeros(B, H, device=device)
        c = torch.zeros(B, H, device=device)

        # first input to decoder is SOS
        prev_token = torch.full((B,), output_stoi[SOS], dtype=torch.long, device=device)

        logits_list = []
        attn_list = []

        T_out = target.size(1) - 1 if target is not None else max_len - 1

        # build mask for attention: (B, 1, T_in) - broadcast over the single query
        attn_mask = input_mask.unsqueeze(1)

        for t in range(T_out):
            # embed previous token
            emb = self.embedding(prev_token).squeeze(1)

            # compute query from current decoder hidden state
            q = self.W_q(h).unsqueeze(1)

            # attention over encoder outputs
            context, attn = scaled_dot_product_attention(q, k, v, attn_mask)
            context = context.squeeze(1)
            # append to attn_list
            attn_list.append(attn.squeeze(1))

            # feed [embedding, context] into the LSTM cell
            lstm_input = torch.cat([emb, context], dim=-1)
            h, c = self.lstm_cell(lstm_input, (h, c))

            # produce output logits using both hidden state and context
            logits = self.out_proj(torch.cat([h, context], dim=-1))  # (B, vocab)

            # append logits to logits list
            logits_list.append(logits)

            # pick next input token (next token from target if it exists, greedy otherwise)
            if target is not None:
                prev_token = target[:, t].unsqueeze(1)
            else:
                prev_token = logits.argmax(dim=-1, keepdim=True)

        logits_stack = torch.stack(logits_list, dim=1)
        attn_stack = torch.stack(attn_list, dim=1)
        return logits_stack, attn_stack


class Seq2SeqAttn(nn.Module):
    def __init__(self, input_vocab_size, output_vocab_size,
                 embed_dim=32, hidden_dim=64):
        super().__init__()
        self.encoder = Encoder(input_vocab_size, embed_dim, hidden_dim)
        self.decoder = AttentionDecoder(output_vocab_size, embed_dim, hidden_dim)

    def forward(self, src, target=None):
        input_mask = (src != input_stoi[PAD]).long() # (B, T_in)
        enc_out = self.encoder(src)
        logits, attn = self.decoder(enc_out, input_mask, target)
        return logits, attn


model = Seq2SeqAttn(len(input_vocab), len(output_vocab)).to(device)
# print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {n_params:,}")
model


Total parameters: 118,638


Seq2SeqAttn(
  (encoder): Encoder(
    (embedding): Embedding(125, 32, padding_idx=0)
    (lstm): LSTM(32, 64, batch_first=True, bidirectional=True)
    (proj): Linear(in_features=128, out_features=64, bias=True)
  )
  (decoder): AttentionDecoder(
    (embedding): Embedding(14, 32, padding_idx=0)
    (lstm_cell): LSTMCell(96, 64)
    (W_q): Linear(in_features=64, out_features=64, bias=True)
    (W_k): Linear(in_features=64, out_features=64, bias=True)
    (W_v): Linear(in_features=64, out_features=64, bias=True)
    (out_proj): Linear(in_features=128, out_features=14, bias=True)
  )
)

In [131]:
src, target, _, _ = next(iter(train_loader))
src, target = src.to(device), target.to(device)
logits, attn = model(src, target)
logits[0], attn[0]

(tensor([[ 3.4766e-02,  4.0184e-02,  6.8617e-02,  8.0905e-02, -1.4738e-02,
          -8.6293e-02, -7.8840e-03,  4.2116e-02,  1.6026e-02, -1.0519e-01,
          -1.0080e-01,  5.5720e-02,  9.6938e-03, -1.0594e-01],
         [ 5.3085e-02,  6.5314e-02,  4.8757e-02,  9.4124e-02, -5.3288e-02,
          -1.0052e-01,  9.2314e-03,  3.7397e-02, -1.1728e-02, -1.0895e-01,
          -1.0895e-01,  6.5681e-02,  2.3623e-03, -1.2939e-01],
         [ 2.3188e-02, -3.6242e-02,  2.3239e-02,  1.2218e-02, -2.3012e-02,
          -5.6375e-02, -8.7651e-02,  1.6948e-02,  3.9952e-02, -1.1251e-01,
          -4.6662e-02,  6.0958e-02, -5.2824e-03, -1.1817e-01],
         [ 8.8704e-03, -9.0873e-02,  4.6700e-02, -2.7193e-03,  1.7648e-02,
          -1.0931e-01, -1.6060e-01, -3.0556e-02,  5.3975e-02, -1.6227e-01,
          -7.8801e-02,  1.1140e-01,  4.4354e-02, -7.7112e-02],
         [-1.7480e-02, -5.1814e-02,  8.1063e-02,  1.7434e-06,  7.3100e-02,
          -5.9210e-02, -1.2649e-01, -8.5083e-02,  3.0760e-02, -1.1624e-01

In [133]:
from tqdm import tqdm

optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index=output_stoi[PAD])

N_EPOCHS = 15

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for src, target, _, _ in tqdm(train_loader):
        src, target = src.to(device), target.to(device)
        logits, _ = model(src)
        # targets for loss: shift by 1 (predict tokens from position 1 onward)
        loss_targets = target[:, 1:]
        loss = loss_fn(
            logits.reshape(-1, logits.size(-1)),
            loss_targets.reshape(-1)
        )
        total_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for src, target, _, target_strings in test_loader:
            src = src.to(device)
            logits, _ = model(src, target=None) # no teacher forcing for inference
            preds = logits.argmax(-1).cpu().tolist()
            for pred_ids, true_str in zip(preds, target_strings):
                chars = []
                for i in pred_ids:
                    if i == output_stoi[EOS]:
                        break
                    if i == output_stoi[PAD]:
                        continue
                    chars.append(output_itos[i])
                pred_str = "".join(chars)
                if pred_str == true_str:
                    correct += 1
                total += 1

    print(f"Epoch {epoch}: train loss = {total_loss/len(train_loader):.4f} | "
          f"test exact-match = {correct}/{total} ({100*correct/total:.1f}%)")

100%|██████████| 209/209 [00:07<00:00, 26.42it/s]


Epoch 1: train loss = 0.1484 | test exact-match = 1145/1486 (77.1%)


100%|██████████| 209/209 [00:06<00:00, 34.80it/s]


Epoch 2: train loss = 0.1233 | test exact-match = 1217/1486 (81.9%)


100%|██████████| 209/209 [00:06<00:00, 29.92it/s]


...

Epoch 13: train loss = 0.0156 | test exact-match = 1380/1486 (92.9%)


100%|██████████| 209/209 [00:06<00:00, 30.66it/s]


Epoch 14: train loss = 0.0157 | test exact-match = 1379/1486 (92.8%)


100%|██████████| 209/209 [00:05<00:00, 35.17it/s]


Epoch 15: train loss = 0.0165 | test exact-match = 1385/1486 (93.2%)


In [136]:
text = input("Enter text: ")

model.eval()

encoded = encode_input(text)
input_tensor = torch.tensor([encoded], dtype=torch.long).to(device)

with torch.no_grad():
    logits, _ = model(input_tensor, target=None)

pred_ids = logits.argmax(-1)[0].cpu().tolist()
chars = []
for i in pred_ids:
    if i == output_stoi[EOS]:
        break

    if i not in (output_stoi[PAD], output_stoi[SOS]):
        chars.append(output_itos[i])

output = "".join(chars)

print(f"Predicted Date: {output}")

Enter text: It was a sunny day on November 13th, 2069.
Predicted Date: 2069-11-13
